# Synthetic synonym datasets

Thin notebook over `scripts/preprocess_synonyms.py` — the loading, filtering, DAS
generation and parquet writing live in the Python module (single source of truth,
shared with `python -m scripts.preprocess_synonyms`). Here we just call them and
inspect the frames.

Builds two streams, each supervising a single head of `doc_classifier_syn`:
- `syn` — one synonym per doc → `dp` (single-label).
- `syn_das` — `k` synonyms concatenated per doc → `das` code list (multi-label).

In [ ]:
import sys

sys.path.insert(0, "..")  # make the repo-root `scripts` package importable

import pandas as pd

from scripts._common import (
    REFERENTIAL,
    get_spark,
    load_cim10_referential,
    save_to_parquet,
)
from scripts.preprocess_synonyms import (
    HDFS_BASE,
    SCRATCH,
    SYNONYMS_PICKLE,
    build_das_frame,
    build_dp_frame,
)

spark = get_spark()

In [ ]:
# Valid CIM-10 codes per head: dp (principal), das (associated).
dp_codes, das_codes, _ = load_cim10_referential(REFERENTIAL)
print(len(dp_codes), len(das_codes))

In [ ]:
syn = pd.read_pickle(SYNONYMS_PICKLE)
syn

## `dp` stream — one synonym per document

In [ ]:
dp_frame = build_dp_frame(syn, dp_codes)
dp_frame

In [ ]:
save_to_parquet(
    spark.createDataFrame(dp_frame),
    hdfs_path=f"{HDFS_BASE}/syn",
    local_path=SCRATCH / "syn",
    columns=["note_text", "dp"],
    num_partitions=121,
)

## `das` stream — k synonyms concatenated per document

In [ ]:
das_frame = build_das_frame(syn, das_codes)
das_frame.head()

In [ ]:
save_to_parquet(
    spark.createDataFrame(das_frame),
    hdfs_path=f"{HDFS_BASE}/syn_das",
    local_path=SCRATCH / "syn_das",
    columns=["note_text", "das"],
    num_partitions=121,
)